# Granite Peak CRE: 02 Ingest Overture Maps

Downloads [Overture Maps](https://overturemaps.org/) **Places**, **Buildings**, **Divisions** (points) and **Division areas** (neighborhood and locality polygons) data for the Granite Peak MSAs using the public GeoParquet release on Azure/AWS. Data is filtered by bounding box and written to the landing volume for later Lakeflow ingest.

**License:** Overture Maps data is released under [CDLA Permissive 2.0](https://cdla.dev/permissive-2-0/) and [ODbL](https://opendatacommons.org/licenses/odbl/) (for OpenStreetMap-derived layers). Attribution is required.

**Risk note:** if outbound HTTP is blocked on serverless, download locally and upload to the volume (see PLAN.md §10).

In [1]:
dbutils.widgets.text("catalog", "smmorgan_catalog", "Catalog")
dbutils.widgets.text("schema", "granite_commercial_real_estate", "Schema")

catalog = dbutils.widgets.get("catalog").strip()
schema  = dbutils.widgets.get("schema").strip()
dbutils.widgets.text("types", "places,buildings,divisions,division_areas", "Layers to download")
LAYERS = [x.strip() for x in dbutils.widgets.get("types").split(",") if x.strip()]
OUT_DIR = f"/Volumes/{catalog}/{schema}/landing/overture"
print(f"Output: {OUT_DIR}  |  Layers: {LAYERS}")

Output: /Volumes/smmorgan_catalog/granite_commercial_real_estate/landing/overture  |  Layers: ['division_areas']


In [2]:
# One generous bounding box that covers all 8 MSAs (Seattle to Phoenix, Boise to Albuquerque)
# rather than 8 individual downloads, to keep the download simple.
BBOX = {
    "min_lon": -123.5,  # west of Seattle/Portland
    "max_lon": -104.5,  # east of Denver/Albuquerque
    "min_lat":  32.5,   # south of Phoenix
    "max_lat":  48.0,   # north of Seattle
}
print(f"Bounding box: {BBOX}")

Bounding box: {'min_lon': -123.5, 'max_lon': -104.5, 'min_lat': 32.5, 'max_lat': 48.0}


In [3]:
# Overture Maps Foundation publishes GeoParquet on both Azure and AWS.
# On AWS-hosted Databricks we use the S3 paths from the latest release.
# See: https://docs.overturemaps.org/getting-data/

OVERTURE_RELEASE = "2026-10-21.0"
OVERTURE_BASE = f"s3://overturemaps-us-west-2/release/{OVERTURE_RELEASE}"

def read_overture_theme(theme, typ):
    """Read a single Overture type filtered to our bounding box."""
    path = f"{OVERTURE_BASE}/theme={theme}/type={typ}"
    print(f"Reading {theme}/{typ} from {path} ...")
    df = (
        spark.read.format("parquet").load(path)
        .filter(f"bbox.xmin >= {BBOX['min_lon']} AND bbox.xmax <= {BBOX['max_lon']}"
                f" AND bbox.ymin >= {BBOX['min_lat']} AND bbox.ymax <= {BBOX['max_lat']}")
    )
    return df

# layer name -> (theme, type, optional extra filter)
LAYER_SPECS = {
    "places":         ("places",    "place",         None),
    "buildings":      ("buildings", "building",      None),
    "divisions":      ("divisions", "division",      None),
    # Polygons for exact neighborhood membership (h3_coverash3 candidates + st_covers)
    "division_areas": ("divisions", "division_area", "subtype IN ('neighborhood', 'macrohood', 'locality')"),
}

try:
    frames = {}
    for layer in LAYERS:
        theme, typ, extra = LAYER_SPECS[layer]
        df = read_overture_theme(theme, typ)
        frames[layer] = df.filter(extra) if extra else df
    print("Overture reads succeeded.")
    overture_ok = True
except Exception as e:
    print(f"\n*** Overture download BLOCKED or failed: {e}")
    print("Falling back: upload Overture extracts manually to the landing volume (see PLAN.md risk row).")
    overture_ok = False

Reading divisions/division_area from s3://overturemaps-us-west-2/release/2026-10-21.0/theme=divisions/type=division_area ...
Overture reads succeeded.


In [4]:
import os, json

if overture_ok:
    meta_path = os.path.join(OUT_DIR, "_overture_meta.json")
    summary = json.load(open(meta_path)).get("row_counts", {}) if os.path.exists(meta_path) else {}
    for name, df in frames.items():
        out = f"{OUT_DIR}/{name}"
        cnt = df.count()
        df.write.mode("overwrite").parquet(out)
        summary[name] = cnt
        print(f"  {name}: {cnt:,} rows -> {out}")

    # Write a metadata sidecar so the pipeline knows the source
    meta = {
        "source": "Overture Maps Foundation",
        "release": OVERTURE_RELEASE,
        "license": "CDLA Permissive 2.0 / ODbL",
        "bbox": BBOX,
        "row_counts": summary,
    }
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)
    print(f"\nMetadata: {meta_path}")
    print(f"Total rows: {sum(summary.values()):,}")
else:
    print("Skipped: Overture data not available. Upload manually.")

  division_areas: 5,494 rows -> /Volumes/smmorgan_catalog/granite_commercial_real_estate/landing/overture/division_areas

Metadata: /Volumes/smmorgan_catalog/granite_commercial_real_estate/landing/overture/_overture_meta.json
Total rows: 34,923,333
